# Import Libraries

In [1]:
import os
import joblib
import numpy as np
import librosa
import tensorflow as tf
import sounddevice as sd
import soundfile as sf
import threading
from datetime import datetime
import tensorflow as tf
import logging
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
tf.get_logger().setLevel('ERROR')  
logging.getLogger('tensorflow').setLevel(logging.ERROR)  
from IPython.display import Audio, display

# Load the Models

In [2]:
MODEL_PATH = "../model/audio_emotion_model.h5"
ENCODER_PATH = "../model/label_encoder.pkl"
SCALER_PATH = "../model/scaler.pkl"

# Testing on new Data

In [3]:
SAVE_DIR = "../Data/Testing Data"
os.makedirs(SAVE_DIR, exist_ok=True) 

SR = 22050
TIMESTEPS = 10

In [4]:
def play_in_notebook(file_path):
    display(Audio(filename=file_path, autoplay=False))

In [5]:
def get_new_record_path():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return os.path.join(SAVE_DIR, f"recorded_voice_{timestamp}.wav")

In [6]:
def extract_features_from_file(path):
    data, _ = librosa.load(path, sr=SR)

    mel = np.mean(librosa.feature.melspectrogram(y=data, sr=SR, n_mels=128).T, axis=0)
    mfcc = np.mean(librosa.feature.mfcc(y=data, sr=SR, n_mfcc=40).T, axis=0)
    chroma = np.mean(librosa.feature.chroma_stft(y=data, sr=SR).T, axis=0)

    pitch, _ = librosa.piptrack(y=data, sr=SR)
    pitch_vals = pitch[pitch > 0]
    pitch_mean = np.mean(pitch_vals) if pitch_vals.size > 0 else 0.0

    features = np.hstack((mel, mfcc, chroma, pitch_mean))

    if features.shape[0] != 181:
        raise ValueError("Invalid feature length")

    return features.astype(np.float32)

In [7]:
def load_resources():
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    encoder = joblib.load(ENCODER_PATH)
    scaler = joblib.load(SCALER_PATH)
    return model, encoder, scaler

In [8]:
def prepare_input_vector(feat181, scaler):
    scaled = scaler.transform(feat181.reshape(1, -1)).reshape(-1)
    feature_180 = scaled[:180]
    per_step = 180 // TIMESTEPS
    return feature_180.reshape(1, TIMESTEPS, per_step, 1)

In [9]:
def predict_emotion(file_path, model, encoder, scaler):
    feat = extract_features_from_file(file_path)
    X = prepare_input_vector(feat, scaler)
    probs = model.predict(X)[0]

    idx = int(np.argmax(probs))
    label = encoder.inverse_transform([idx])[0]
    prob = float(probs[idx])

    print(f"File: {file_path}")
    print(f"Predicted Emotion: {label}")

In [10]:
def play_audio(filepath, sr=SR):
    try:
        if _IPY_AVAILABLE:
            ipy_display(IPyAudio(filename=filepath, rate=sr))
            return
    except Exception:
        pass

    try:
        data, file_sr = sf.read(filepath, dtype='float32')
        if data.ndim > 1:
            data = data.mean(axis=1)
        if file_sr != sr:
            data = librosa.resample(data, orig_sr=file_sr, target_sr=sr)
        sd.play(data, samplerate=sr)
        sd.wait()
    except Exception as e:
        print(f"Unable to play audio: {e}")

In [11]:
def _record_loop():
    return

def record_audio():
    global recording, is_recording
    recording = []

    input("Press ENTER to start recording...")
    print("Recording... Press ENTER again to stop.")

    stop_event = threading.Event()

    def wait_for_enter():
        try:
            input()
        except Exception:
            pass
        stop_event.set()

    waiter = threading.Thread(target=wait_for_enter, daemon=True)
    waiter.start()

    is_recording = True
    block_frames = int(0.5 * SR)  

    try:
        with sd.InputStream(samplerate=SR, channels=1, dtype='float32') as stream:
            while not stop_event.is_set():
                frames, overflowed = stream.read(block_frames)
                if overflowed:
                    pass
                recording.append(frames.copy())
    except Exception as e:
        print(f"Recording failed: {e}")
        is_recording = False
        stop_event.set()

    is_recording = False
    waiter.join(timeout=0.1)

    if len(recording) == 0:
        print("No audio captured.")
        return None

    audio = np.concatenate(recording, axis=0).squeeze()

    max_val = np.max(np.abs(audio)) if audio.size else 0.0
    if max_val > 0:
        audio = audio / max_val

    rec_file = get_new_record_path()
    try:
        sf.write(rec_file, audio, SR)
        print(f"\nSaved recording to: {rec_file}")
        return rec_file
    except Exception as e:
        print(f"Failed to save recording: {e}")
        return None


In [14]:
model, encoder, scaler = load_resources()

print("\nChoose an option:")
print("1. Enter path to audio file")
print("2. Record your voice")

choice = input("Your choice (1/2): ").strip()

if choice == "1":
    path = input("Enter full audio file path: ").strip()
    if os.path.exists(path):
        play_in_notebook(path)
        predict_emotion(path, model, encoder, scaler)
    else:
        print("File not found!")

elif choice == "2":
    rec_path = record_audio()
    play_in_notebook(rec_path) 
    predict_emotion(rec_path, model, encoder, scaler)

else:
    print("Invalid choice! Please select 1 or 2.")


Choose an option:
1. Enter path to audio file
2. Record your voice


Your choice (1/2):  1
Enter full audio file path:  ..\Data\SAVEE data\JE\n19.wav


1/1 [==============================] - 1s 1s/step
File: ..\Data\SAVEE data\JE\n19.wav
Predicted Emotion: neutral
